# Can we predict 30-day readmission?
Data: `data.csv` (one row per hospital stay), dictionary in `columns.csv`. Target: `readmit_30d`.

Key design decision: the same person appears in many rows (stays). Random row-level splits would put the same person in train and test, so every evaluation here uses **group-wise splits by `person_key`**. `person_key` itself is never used as a feature.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, GroupShuffleSplit, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(df.shape); print(pd.read_csv('columns.csv').to_string(index=False))

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(),'missing;', df.duplicated().sum(),'duplicate rows;', df.person_key.nunique(),'persons;', df.person_key.value_counts().median(),'median stays/person')
print(df.readmit_30d.value_counts(normalize=True).rename('share'))
print(df.describe().T)
# Are person-level attributes constant within a person?
print(df.groupby('person_key')[['birth_year','sex','postcode_area']].nunique().max())
print(df.groupby('person_key').visit_number.apply(lambda s: s.is_unique).all(), 'visit_number unique within person')

## 2. Univariate look

In [ ]:
num=['visit_number','birth_year','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,col in zip(ax.ravel(),num):
    q=pd.qcut(df[col],10,duplicates='drop'); g=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(g)),g.values,'o-'); a.axhline(df.readmit_30d.mean(),ls='--',c='grey'); a.set_title(col); a.set_xlabel('decile'); a.set_ylim(0,0.5)
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())
print(df[num+['readmit_30d']].corr()['readmit_30d'].drop('readmit_30d').round(3))

## 3. Models, evaluated with grouped 5-fold CV
Baselines: constant prior; logistic regression; gradient boosting. Metrics: ROC AUC and average precision (prevalence is ~20%, so the AP of a useless model is ~0.20).

In [ ]:
y=df.readmit_30d.values; groups=df.person_key.values
X=df.drop(columns=['readmit_30d','person_key']).copy()
num=[c for c in X.columns if c!='sex']
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(drop='if_binary'),['sex'])])
X['sex']=X.sex.astype('category')
models={'prior':DummyClassifier(strategy='prior'),
 'logreg':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'hgb':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,categorical_features='from_dtype',random_state=0)}
cv=StratifiedGroupKFold(5,shuffle=True,random_state=0)
oof={}; rows=[]
for n,m in models.items():
    p=cross_val_predict(m,X,y,cv=cv,groups=groups,method='predict_proba')[:,1]; oof[n]=p
    rows.append((n,roc_auc_score(y,p),average_precision_score(y,p),brier_score_loss(y,p)))
res=pd.DataFrame(rows,columns=['model','AUC','AP','Brier']).set_index('model'); res

Uncertainty: bootstrap over **persons** (rows within a person are correlated) and per-fold AUCs.

In [ ]:
rng=np.random.default_rng(0)
pid=pd.Series(np.arange(len(df))).groupby(groups).apply(np.array); keys=pid.index.values
def boot(p,B=300):
    out=[]
    for _ in range(B):
        idx=np.concatenate(pid[rng.choice(keys,len(keys))].values); out.append(roc_auc_score(y[idx],p[idx]))
    return np.percentile(out,[2.5,97.5])
for n in ['logreg','hgb']: print(n,'AUC 95% CI',boot(oof[n]).round(3))
print('logreg per-fold AUC',[round(roc_auc_score(y[te],oof['logreg'][te]),3) for _,te in cv.split(X,y,groups)])

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n in ['logreg','hgb']:
    f,t,_=roc_curve(y,oof[n]); ax[0].plot(f,t,label=f"{n} ({res.AUC[n]:.3f})")
    pr,rc,_=precision_recall_curve(y,oof[n]); ax[1].plot(rc,pr,label=n)
    fp,mp=calibration_curve(y,oof[n],n_bins=10,strategy='quantile'); ax[2].plot(mp,fp,'o-',label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set_title('ROC'); ax[1].axhline(y.mean(),c='k',ls=':'); ax[1].set_title('Precision-recall')
ax[2].plot([0,.5],[0,.5],'k:'); ax[2].set_title('Calibration (out-of-fold)'); ax[2].set_xlabel('predicted'); ax[2].set_ylabel('observed')
for a in ax: a.legend()
plt.tight_layout(); plt.show()

## 4. What drives the prediction?

In [ ]:
tr,te=next(GroupShuffleSplit(1,test_size=0.25,random_state=1).split(X,y,groups))
m=models['hgb'].fit(X.iloc[tr],y[tr]); print('holdout AUC (unseen persons):',round(roc_auc_score(y[te],m.predict_proba(X.iloc[te])[:,1]),3))
pi=permutation_importance(m,X.iloc[te],y[te],scoring='roc_auc',n_repeats=20,random_state=0)
imp=pd.Series(pi.importances_mean,X.columns).sort_values()
imp.plot.barh(xerr=pd.Series(pi.importances_std,X.columns)[imp.index]); plt.xlabel('AUC drop when shuffled'); plt.show()
lr=models['logreg'].fit(X.iloc[tr],y[tr]); print(pd.Series(lr[-1].coef_[0],lr[0].get_feature_names_out()).sort_values().round(3))

## 5. Sanity check: does a naive split overstate performance?
Row-level random CV (ignoring that persons repeat) versus grouped CV, same model.

In [ ]:
p=cross_val_predict(models['hgb'],X,y,cv=StratifiedKFold(5,shuffle=True,random_state=0),method='predict_proba')[:,1]
print('naive row-split AUC:',round(roc_auc_score(y,p),3),'| grouped AUC:',round(res.AUC['hgb'],3))

## 6. Conclusion
Numbers below are from the run when this notebook was built (re-run to refresh).

- **Signal exists but is weak.** Grouped-CV AUC ≈ 0.65 for logistic regression (95% CI ≈ 0.63–0.66), ≈ 0.64 for gradient boosting, vs 0.50 for the prior. AP ≈ 0.32 vs 0.20 base rate. That ranks readmitters better than chance, but is not strong enough to act on for individual patients (a 'high-risk' flag would be wrong most of the time).
- **Drivers:** longer stay, more discharge medications and higher `lab_a` raise risk; `postcode_area` shows a small negative association. Demographics, BMI, `lab_b` and `visit_number` contribute little. The simple linear model is as good as boosting, so there is no evidence of strong non-linear/interaction structure in these features.
- **Evaluation caveat:** a naive row-level split gives AUC ≈ 0.68 versus ≈ 0.64 grouped, because the same people appear in train and test. Use the grouped number.
- **Limits:** only 1,500 people; no prior-admission history, diagnoses, or discharge disposition, which are usually the strongest readmission predictors. `length_of_stay_days` and `n_medications` are known at discharge, which fits a discharge-time risk score but not an admission-time one. Calibration is acceptable for logistic regression but with a narrow range of predicted risk.
- **Next steps:** add history features (prior stays' outcomes/lengths per person, time since last stay), check the `postcode_area` effect as a categorical, and validate on a later time period if dates become available.